# Stage II (lesions) - 02. Training HSMoE-AUNet

Trains the HSMoE-AUNet of one lesion type on its training split and keeps the checkpoint with
the best validation Dice. Every lesion-specific setting comes from
`hierarchiretina.stage2.hsmoe_aunet.LESION_CONFIGS`:

| | MA | HE | EX | CWS |
|---|---|---|---|---|
| Expert types | 4 base | + multi-scale | + multi-scale, top-hat, DoG | + multi-scale, top-hat, soft-region, texture |
| Experts skip / bottleneck / decoder | 6/8/10, 12, 10/8/6 | 6/8/10, 12, 10/8/6 | 7/9/11, 14, 11/9/7 | 8/10/12, 16, 12/10/8 |
| FTL a / b / gamma | .30/.70/2 | .50/.50/4/3 | .45/.55/1 | .55/.45/4/3 (1 in warm-up) |
| Extra loss terms | - | Sobel boundary 0.3 | Sobel+Laplacian 0.5, optic disc 0.15 | soft boundary 0.15, region 0.02, optic disc 0.08 |
| BCE weight | 0.1 | 0.1 | 0.1 | 0.2, pos. weight 10 |
| Learning rate / schedule | 1e-4, plateau | 1e-4, cosine restarts (20, x2) | 1e-4, cosine (400) | 7e-5, cosine restarts (15, x2) |
| Batch x accumulation | 4 x 4 | 2 x 4 | 2 x 4 | 2 x 4 |
| Gradient clip | 1.0 | 1.0 | 1.0 | 0.5 (+ non-finite loss guard) |

Common: ConvNeXt-Small encoder (`convnext_small.in12k_ft_in1k`, first 20 parameter tensors
frozen), whole 1024 x 1024 images, AdamW (weight decay 1e-4), mixed precision, at most 400
epochs, early stopping after 40 epochs without improvement of the validation Dice (threshold
0.5). The CWS loss is introduced in three phases (epochs 0-14, 15-29, >= 30).

1. **Configuration**  2. **Data loaders**  3. **Model and loss**  4. **Training**

## Step 1. Configuration

In [ ]:
from pathlib import Path

LESION = "MA"  # "MA" | "HE" | "EX" | "CWS"

DATA_ROOT = Path("data/stage2_lesions")
MANIFEST = DATA_ROOT / LESION / "manifest.csv"          # written by notebook 01
CHECKPOINT_DIR = Path("checkpoints/stage2/hsmoe_aunet") / LESION
NUM_WORKERS = 0      # the original runs used 0 (Windows)
SEED = 42

In [ ]:
import torch

from hierarchiretina.stage2.engine_lesion import configure_cuda_backends, fit, set_seed
from hierarchiretina.stage2.hsmoe_aunet import (build_hsmoe_aunet, count_parameters,
                                                get_lesion_config)
from hierarchiretina.stage2.lesion_data import build_dataloaders, load_splits
from hierarchiretina.stage2.lesion_losses import build_lesion_loss

set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    configure_cuda_backends()
cfg = get_lesion_config(LESION)
print(DEVICE, "|", cfg.name, "| lr", cfg.lr, "| scheduler", cfg.scheduler, cfg.scheduler_kwargs)

## Step 2. Data loaders

In [ ]:
train_df, val_df, test_df = load_splits(MANIFEST, LESION)
train_loader, val_loader, _ = build_dataloaders(LESION, train_df, val_df, test_df,
                                                num_workers=NUM_WORKERS)
print(f"train {len(train_df)} ({len(train_loader)} batches of {cfg.batch_size}) | "
      f"val {len(val_df)} | test {len(test_df)}")

## Step 3. Model and loss

`pretrained=True` loads the ImageNet-12k/1k ConvNeXt-Small weights through timm. As in the
paper runs, the model's weight initialisation then re-initialises the encoder's convolution and
linear weights (see README, Reproducibility notes); this is kept so results and checkpoints match.

In [ ]:
model = build_hsmoe_aunet(LESION, pretrained=True)
criterion = build_lesion_loss(LESION)
total, trainable = count_parameters(model)
print(f"parameters: {total / 1e6:.2f} M (paper {cfg.paper_params_m} M), "
      f"trainable {trainable / 1e6:.2f} M")

## Step 4. Training

Interrupted runs resume from `last_checkpoint.pth`. The best model is written to
`best_model.pth` (`{'model_state', 'epoch', 'val_dice'}`), the per-epoch log to
`training_log.csv`.

In [ ]:
result = fit(model, criterion, train_loader, val_loader, LESION, CHECKPOINT_DIR, DEVICE)
print(f"best validation Dice {result['best_dice']:.4f} at epoch {result['best_epoch']} "
      f"(paper run: epoch {cfg.paper_best_epoch})")